In [ ]:
import numpy as np
import pandas as pd

# =========================================================================
# STEP 1: LOAD & PREPARE DATASET
# =========================================================================
try:
    df = pd.read_csv(r"\Users\suloc\Downloads\housing_price.csv")
    feature_columns = ['SquareFootage', 'Bedrooms', 'Bathrooms', 'Age']
    target_column = 'Price'
    
    X_raw = df[feature_columns].values.astype(float)
    y = df[target_column].values.reshape(-1, 1).astype(float)
except FileNotFoundError:
    print("⚠️ 'housing_data.csv' not found. Creating dummy dataset...")
    X_raw = np.array([
        [2104, 3, 2, 15], [1600, 3, 2, 20], [2400, 3, 3, 10], 
        [1416, 2, 1, 23], [3000, 4, 3, 8],  [1985, 4, 2, 12]
    ], dtype=float)
    y = np.array([399.9, 329.9, 369.0, 232.0, 539.9, 299.9], dtype=float).reshape(-1, 1)

m, n = X_raw.shape

# Feature Scaling (Crucial for Regularization so weights are penalized equally!)
mu = np.mean(X_raw, axis=0)
sigma = np.std(X_raw, axis=0)
X_scaled = (X_raw - mu) / sigma

# Add Column of 1s for Bias Term (theta_0)
X = np.hstack((np.ones((m, 1)), X_scaled))


# =========================================================================
# STEP 2: RIDGE REGRESSION (L2 Regularization)
# =========================================================================
def ridge_gradient_descent(X, y, l2_lambda=10.0, alpha=0.01, iterations=1500):
    m, n = X.shape
    theta = np.zeros((n, 1))
    
    for _ in range(iterations):
        predictions = np.dot(X, theta)
        errors = predictions - y
        
        # Calculate standard OLS gradient
        gradients = (1 / m) * np.dot(X.T, errors)
        
        # Add L2 penalty gradient: lambda * theta (Do NOT penalize bias term theta_0!)
        l2_penalty = (l2_lambda / m) * theta
        l2_penalty[0] = 0  # Zero out bias penalty
        
        # Simultaneous parameter update
        theta = theta - alpha * (gradients + l2_penalty)
        
    return theta


# =========================================================================
# STEP 3: LASSO REGRESSION (L1 Regularization via Sub-gradient)
# =========================================================================
def lasso_gradient_descent(X, y, l1_lambda=10.0, alpha=0.01, iterations=1500):
    m, n = X.shape
    theta = np.zeros((n, 1))
    
    for _ in range(iterations):
        predictions = np.dot(X, theta)
        errors = predictions - y
        
        # Calculate standard OLS gradient
        gradients = (1 / m) * np.dot(X.T, errors)
        
        # Add L1 penalty gradient: lambda * sign(theta) (Do NOT penalize theta_0!)
        l1_penalty = (l1_lambda / m) * np.sign(theta)
        l1_penalty[0] = 0  # Zero out bias penalty
        
        # Simultaneous parameter update
        theta = theta - alpha * (gradients + l1_penalty)
        
    return theta


# =========================================================================
# STEP 4: EXECUTION & WEIGHT COMPARISON
# =========================================================================
# Run unconstrained OLS vs. Ridge vs. Lasso with lambda = 10.0
theta_ols = ridge_gradient_descent(X, y, l2_lambda=0.0, alpha=0.01, iterations=1500)
theta_ridge = ridge_gradient_descent(X, y, l2_lambda=10.0, alpha=0.01, iterations=1500)
theta_lasso = lasso_gradient_descent(X, y, l1_lambda=10.0, alpha=0.01, iterations=1500)

print("=" * 65)
print("  MODEL WEIGHT COMPARISON Across Features")
print("=" * 65)
feature_names = ['Bias (Intercept)', 'SquareFootage', 'Bedrooms', 'Bathrooms', 'Age']

for i, name in enumerate(feature_names):
    print(f"{name:<18} | OLS: {theta_ols[i][0]:8.3f} | Ridge (L2): {theta_ridge[i][0]:8.3f} | Lasso (L1): {theta_lasso[i][0]:8.3f}")

print("=" * 65)


# =========================================================================
# STEP 5: PREDICTION DEMO
# =========================================================================
# Sample: [2200 sqft, 3 bedrooms, 2 bathrooms, 10 yrs old]
new_sample = np.array([2200, 3, 2, 10], dtype=float)
new_sample_scaled = (new_sample - mu) / sigma
new_sample_input = np.append([1], new_sample_scaled).reshape(1, -1)

pred_ridge = np.dot(new_sample_input, theta_ridge)[0][0]
pred_lasso = np.dot(new_sample_input, theta_lasso)[0][0]

print(f"\n🎯 Ridge Prediction ($L_2$): ${pred_ridge:.2f}k")
print(f"🎯 Lasso Prediction ($L_1$): ${pred_lasso:.2f}k")